# 📘 Example Driver Test Notebook

This notebook provides a **comprehensive, sequential test** of all functions in the `Example` instrument driver class hierarchy. It serves as a template for creating functional test notebooks for new drivers.

| Layer | Purpose | Key Methods |
|---|---|---|
| 1 | **Base Instrument** (`Instrument`) | `idn()` |
| 2 | **Generic Interface** (`Example`) | `output()`, `set_mode()`, `set_voltage()`, `set_current()`, `configure_instrument()`, `quick_read()`, `set_optional()` |
| 3 | **Specific Driver** (e.g. `AnnotatedGenericDriver`) | Implements Level 2 + inherits `Scpi` commands (`reset()`, `clear()`, etc.) |

---

**Instructions for the Technician:**
1. Ensure an instrument (or virtual simulation) is available.
2. Run each cell **sequentially** from top to bottom.
3. Verify the **Expected Result** and the **✅ PASS** criteria in the markdown following each cell.
4. Note any failures before proceeding.

---
## 0. Setup & Connection
This section handles the connection to the instrument using PIEC's autodetect system.

In [ ]:
import numpy as np
import pandas as pd
from piec.drivers.autodetect import autodetect

In [ ]:
# Change "example" to "VIRTUAL" if no hardware is connected
example = autodetect("example", verbose=True)

**Expected Result:** The cell should print connection info. If successful, the `example` variable should be an instance of a level 3 driver (like `AnnotatedGenericDriver`).

✅ **PASS** if the printed model confirms connection.

---
## 1. Instrument-Level Tests (`Instrument` base class)
Testing the most basic communication: identity check.

### 1.1 Identification (`idn`)

In [ ]:
idn_response = example.idn()
print(f"IDN Response: {idn_response}")

**Expected Result:** A comma-separated ID string (e.g., `Manufacturer, Model, S/N, Ver`).

✅ **PASS** if the identity string represents your instrument.

---
## 2. SCPI-Level Tests (Inherited via `Scpi`)
These methods are available to instruments that inherit from the `Scpi` communication protocol class. If your instrument is non-SCPI, these cells will only work if you implement standard stubs for them.

In [ ]:
try:
    example.reset()                    # *RST
    print("Reset: OK")

    example.clear()                    # *CLS
    print("Clear: OK")

    print("Error:", example.error())   # *ESR? — expect 0
    print("Self-test:", example.self_test())  # *TST? — expect 0
    print("OPC:", example.operation_complete())  # *OPC? — expect 1

    example.initialize()               # reset + clear
    print("Initialize: OK")
except AttributeError:
    print("Skipping SCPI tests: Instrument is non-SCPI and does not have standard SCPI stubs.")

**Expected Result:** All standard SCPI commands should be responsive.

✅ **PASS** if commands succeed or are correctly identified as not applicable.

---
## 3. Generic Driver-Specific Tests (`Example` interface)
Testing the methods defined as the core interface in `example.py`.

### 3.1 Output Control (`output`)

In [ ]:
example.output(channel=1, on=True)
print("Output turned ON on Channel 1.")

example.output(channel=1, on=False)
print("Output turned OFF on Channel 1.")

**Expected Result:** Output should toggle visible state on the hardware.

✅ **PASS** if hardware indicator toggles.

### 3.2 Operating Mode (`set_mode`)

In [ ]:
example.set_mode(channel=1, mode='SWEEP')
print("Mode set to SWEEP.")

example.set_mode(channel=1, mode='CONSTANT')
print("Mode set to CONSTANT.")

**Expected Result:** Mode indicator on screen should change from CONST to SWEEP or equivalent.

✅ **PASS** if the unit changes its mode readout.

### 3.3 Set Voltage (`set_voltage`)

In [ ]:
example.set_voltage(channel=1, voltage=5.0)
print("Voltage set to 5.0 V.")

**Expected Result:** Voltage reading on the device should show 5.000 V.

✅ **PASS** if hardware display matches 5 V.

### 3.4 Combined Config (`configure_instrument`)

In [ ]:
example.configure_instrument(
    channel=1,
    voltage=3.3,
    current=0.05,
    mode='CONSTANT',
    on=True
)
print("Configured instrument to 3.3 V, 50 mA, CONSTANT, enabled.")

**Expected Result:** All settings should update simultaneously on the hardware screen.

✅ **PASS** if all sub-components (V, I, Mode, State) are correct.

### 3.5 Quick Measurement (`quick_read`)

In [ ]:
# Perform a simulated/real measurement
value = example.quick_read(channel=1)
print(f"Measured: {value} units")

**Expected Result:** Should return a numeric value.

✅ **PASS** if the return type is numeric/float.

---
## 4. Optional Driver Features
Testing features that are not required for every instrument but are available in this driver interface.

### 4.1 Set Optional Property (`set_optional`)

In [ ]:
example.set_optional(optional_param="Test string")
print("Optional property set to 'Test string'.")

**Expected Result:** Calls the @optional stub in `example.py`. Since no constraints are defined on Layer 2, it will pass through to the Level 3 driver implementation.

✅ **PASS** if no error is raised.

---
## 5. Cleanup
Final reset to ensure a clean state for the next user.

In [ ]:
if hasattr(example, 'reset'):
    example.reset()
    print("Instrument reset.")
else:
    print("Skipping reset (not available).")
print("Tests complete.")

---
## Test Summary

| Section | Method(s) Tested | Function | Pass/Fail |
|---|---|---|---|
| **1.1** | Identification | `idn()` | |
| **2** | SCPI Suite | `reset()`, `clear()`, `error()`, etc. | |
| **3.1** | Output | `output()` | |
| **3.2** | Mode | `set_mode()` | |
| **3.3** | Voltage | `set_voltage()` | |
| **3.4** | Combined Config | `configure_instrument()` | |
| **3.5** | Measurement | `quick_read()` | |
| **4.1** | Optional Feature | `set_optional()` | |
| **5** | Cleanup | `reset()` | |

**Technician Signature:** _______________  
**Date:** _______________  
**Instrument Serial #:** _______________